# Final GeoParquet

This notebook reads cached GeoParquet data and renders a simple station-by-station timeseries plot.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from IPython.display import Image, display

REPO_ROOT = Path('/home/lcd/git-repos/opendata-parquet')
final_root = Path(os.environ.get('DMI_GEO_PARQUET_FINAL_ROOT', REPO_ROOT / 'data' / 'dmi'))
plot_root = Path(os.environ.get('DMI_GEO_PARQUET_PLOTS', 'tmp/plots'))

def load_final_frame(root: Path) -> pd.DataFrame:
    paths = sorted(root.rglob('*.parquet'))
    if not paths:
        raise FileNotFoundError(f'no parquet files found under {root}')
    frames = [pd.read_parquet(path) for path in paths]
    return pd.concat(frames, ignore_index=True)

def parameter_to_data_array(frame: pd.DataFrame, parameter_id: str) -> xr.DataArray:
    subset = frame[frame['parameter_id'] == parameter_id].copy()
    if subset.empty:
        raise ValueError(f'no rows found for {parameter_id}')
    pivot = subset.pivot_table(
        index='observed_at', columns='station_id', values='value', aggfunc='first'
    ).sort_index()
    pivot.index = pd.to_datetime(pivot.index, utc=True)
    pivot.columns = pivot.columns.astype(str)
    return xr.DataArray(
        pivot.to_numpy(),
        dims=('observed_at', 'station_id'),
        coords={'observed_at': pivot.index, 'station_id': pivot.columns},
        name=parameter_id,
    )

def plot_parameter(frame: pd.DataFrame, parameter_id: str, output_dir: Path) -> Path:
    da = parameter_to_data_array(frame, parameter_id)
    da.plot.line(x='observed_at', hue='station_id')
    fig = plt.gcf()
    ax = fig.axes[0]
    ax.set_title(parameter_id.replace('_', ' ').title())
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f'{parameter_id}.png'
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    return output_path

frame = load_final_frame(final_root)
frame['observed_at'] = pd.to_datetime(frame['observed_at'], utc=True)
frame

In [ ]:
frame.head()

In [ ]:
written = [
    plot_parameter(frame, parameter_id, plot_root)
    for parameter_id in sorted(frame['parameter_id'].dropna().unique())
]
written

In [ ]:
if written:
    display(Image(filename=str(written[0])))